Crea un layer personalizzato chiamato SimpleScalerLayer. A differenza di un layer denso, questo layer deve avere un solo peso scalare addestrabile (una singola variabile). Nel metodo call, moltiplica l'intero input per questo scalare e applica la funzione di attivazione ReLu.
Suggerimento: in add_weight, usa shape=() per creare uno scalare invece di una matrice

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

#1.DEFINIZIONE DEL LAYER PERSONALIZZATO
#Ereditando della classe base 'Layer' per ottenre tutte le funzionalità di un layer standard di TensorFlow
class SimpleScalerLayer(layers.Layer):
    def __init__(self, units=32,**kwargs): #32=numero di neuroni che andremmo a creare in questo layer
        #Inizializzazione: definiamo gli iperparametri del layer (es. n. di neuroni)
        #Chiamiamo super() per permettere a Keras di gestire correttamente il layer
        super(SimpleScalerLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        """
        Definiamo w e b, gli si aggiungono dei pesi, che dovreanno gestire una shape, questi parametri sono aggiornabili
        
        Il metodo build viene eseguito automaticamente la prima volta che il layer viene chiamato in input
        Serve per creare i pesi solo quando conosciamo la dimensione dell'input (input_shape)        """
        #Creazione dei pesi del layer
        #Creazione del 'kernel' (matrice dei pesi):
        #La forma è (numero_caratteristiche_input,unita_output)
        self.scale=self.add_weight(
            shape=(), #scalare: un solo valore come richiesto dall'esercizio, ma in generale potrebbe essere una matrice di pesi se avessimo più unità di output shape=(input_shape[-1], self.units)
            initializer="ones", #inizializzazione dei pesi con una distribuzione normale casuale 
            trainable=True, #indica a TensorFlow di calcolare i gradienti per questo peso
            name='scale') #nome del peso, utile per il debug e la visualizzazione dei pesi del modello



    def call(self, inputs):
        """ 
        Il metodo call contiene la logica del 'Forwar Pass'.
        Qui definiamo l'operazione matematica da eseguire sui dati in ingresso (input), utilizzando i pesi (w) e il bias (b) definito nel metodo build.
        """
        #Definizione della logica di calcolo del layer
        return tf.nn.relu(inputs * self.scale)
        #la classica y=X*W+B, dove X è l'input, W sono i pesi e B è il bias


# ---- FASE  DI VERIFICA (test) DEL LAYER PERSONALIZZATO

#Creiamo un tensore di esempio (batch_size=1, features=3)
input_data=tf.constant([[-1.0,2.0,3.0]],dtype=tf.float32)

#Istanziamo il layer con 4 unità di output
layer = SimpleScalerLayer()

#Applichiamo il layer ai dati di input (questo invocherà internameto build e call)
output = layer(input_data)

#Visualizzazione dei risultati per verificare la trasformazione dei dati attraverso il layer personalizato
print("Forma dell'input:",input_data.shape)
print("Forma dell'output:",output.shape)
print("Risultato dell'output:",output.numpy())


# ---- INTEGRAZIONE DEL LAYER PERSONALIZZATO IN UN MODELLO KERAS


#2.CREAZIONE DEL MODELLO UTILIZZANDO IL LAYER PERSONALIZZATO

#Dimostriamo che il layer personalizzato sia perfettamente compatibile con gli
#altri layer standard di Keras (Dense,Activation, ecc.) e che possa essere utilizzato 
#in un modello sequenziale standard di Keras.


#definiamo il nostro layer in un modello, un semplice sequential con il layer personalizzato 
#seguito da layer standard di Kersas
model = models.Sequential([
    layers.Input(shape=(3 ,)),  #Definiamo la dimensione dell'input (3 caratteristiche in ingresso)
    SimpleScalerLayer(),       #Layer personalizzato
    layers.Dense(1)             #Output layer con 1 unità, layer finale per la regressione (o classificazione binaria)
])

#Mostriamo l'architettura: noteremo i pesi del nostro layer nel conteggio dei parametri del modello
model.summary()

#3.COMPILAZIONE DEL MODELLO
model.compile(optimizer='adam', loss='mse')
#4.GENERAZIONE DI DATI DI ESEMPIO
X_train = np.random.rand(100, 3)  # 100 campioni, 3 caratteristiche ciascuno
y_train = np.random.rand(100, 1)   # 100 campioni, 1 target ciascuno
#5.ADDENDA DEL MODELLO
model.fit(X_train, y_train, epochs=5)
#6.PREDIZIONE CON IL MODELLO
X_test = np.random.rand(10, 3)  # 10 campioni di test
predictions = model.predict(X_test)
print(predictions)




Forma dell'input: (1, 3)
Forma dell'output: (1, 3)
Risultato dell'output: [[0. 2. 3.]]


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_scaler_layer_9           │ (None, 3)              │             1 │
│ (SimpleScalerLayer)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5 (20.00 B)

 Trainable params: 5 (20.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.5818  
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5624 
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5440 
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.5262 
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5089 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
[[0.41024822]
 [0.9960733 ]
 [1.7274147 ]
 [1.3891747 ]
 [1.2977065 ]
 [0.38155627]
 [0.8969484 ]
 [1.4121599 ]
 [1.6884158 ]
 [2.0348275 ]]
